<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">Los Pipelines: Flujo Completo</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">Un pipeline permite encadenar varias transformaciones y un modelo en un único flujo reproducible. En este notebook verás cómo combinar tratamiento de columnas numéricas y categóricas con el entrenamiento de un clasificador sin separar manualmente cada paso.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">Cuando el preprocesamiento y el modelo se coordinan dentro de una sola estructura, disminuye el riesgo de inconsistencias y fugas de información.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A usar ColumnTransformer para tratar distintos tipos de columnas, aplicar escalado y codificación, encapsular todo en un Pipeline y entrenar el flujo completo de una sola vez.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">Si las transformaciones que aplicas en entrenamiento no se repiten igual en inferencia o en test, el modelo deja de ser fiable.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">Un pipeline organiza una cadena ordenada donde cada paso prepara los datos para el siguiente hasta llegar al modelo final.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Creación del dataset
"""Este bloque crea un dataset con columnas numéricas y categóricas.
Este bloque importa "pandas" para trabajar con datos estructurados en forma de DataFrame.
La variable "df" almacena instancias que serán procesadas por el pipeline.
El dataset sirve como entrada para las etapas encadenadas."""

import pandas as pd

df = pd.DataFrame({
    "edad": [22, 35, 47, 29, 41],
    "ingresos": [1500, 2300, 3200, 1800, 2900],
    "ciudad": ["A", "B", "A", "C", "B"],
    "compra": [0, 1, 1, 0, 1]
})

df # Muestra el dataset creado para verificar su contenido.


In [ ]:
# Bloque 2 — Separación de features y label
"""Este bloque separa las características y la etiqueta.
La variable "X" almacena las columnas de entrada.
La variable "y" almacena la columna objetivo.
La separación permite definir el material para el pipeline."""

X = df[["edad", "ingresos", "ciudad"]]
y = df["compra"]

X, y # Muestra las variables X e y para verificar su contenido.


In [ ]:
# Bloque 3 — Definición de transformaciones
"""Este bloque define transformaciones para el pipeline.
Este bloque importa "StandardScaler" y "OneHotEncoder" para estandarización y codificación one-hot, respectivamente.
Este bloque importa "ColumnTransformer" para aplicar transformaciones a las columnas específicas.

Aplicamos la clase "StandardScaler()" a la variable "transformador_num" para crear un objeto estandarizador de columnas numéricas.
Los parámetros "with_mean=True" y "with_std=True" indican que se debe centrar y escalar la variable numérica.
Aplicamos la clase "OneHotEncoder()" a la variable "transformador_cat" para crear un objeto codificador de columnas categóricas robusto ante categorías no vistas.
El parámetro "handle_unknown="ignore"" indica que se deben ignorar las categorías no vistas durante la codificación one-hot.

La clase "ColumnTransformer" se utiliza para aplicar las transformaciones definidas a las columnas correspondientes.
El argumento "transformers" de "ColumnTransformer" es una lista de tuplas, donde cada tupla contiene el nombre de la transformación, 
el objeto transformador y la lista de columnas a las que se aplicará la transformación.

La variable "preprocesamiento" almacena las transformaciones definidas para su posterior uso en el pipeline."""

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

transformador_num = StandardScaler(with_mean=True, with_std=True)
transformador_cat = OneHotEncoder(handle_unknown="ignore")

preprocesamiento = ColumnTransformer([
    ("num", transformador_num, ["edad", "ingresos"]),
    ("cat", transformador_cat, ["ciudad"])
])

preprocesamiento # Muestra el objeto preprocesamiento para verificar su contenido.


In [ ]:
# Bloque 4 — Construcción del pipeline
"""Este bloque construye el pipeline completo.
Este bloque importa "LogisticRegression" para crear un modelo de regresión logística y "Pipeline" para encadenar preprocesamiento y modelo.

La clase "Pipeline()" permite encadenar múltiples pasos de procesamiento y modelado en un solo objeto.
El argumento "steps" de la clase "Pipeline()" es una lista de tuplas.
La primera tupla contiene el nombre de la etapa de preprocesamiento y el objeto que realiza la transformación.
La segunda tupla contiene el nombre de la etapa del modelo y el objeto que realiza la predicción; en este caso es un modelo de regresión logística.
La variable "pipeline" encadena preprocesamiento y modelo.
El pipeline garantiza que todas las etapas se ejecuten en orden."""

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("preprocesamiento", preprocesamiento),
    ("modelo", LogisticRegression())
])

pipeline # Muestra el objeto pipeline para verificar su contenido.

In [ ]:
# Bloque 5 — Entrenamiento y evaluación
"""Este bloque entrena y evalúa el pipeline.
Este bloque importa "train_test_split" para dividir los datos en conjuntos de entrenamiento y prueba.
Este bloque importa "accuracy_score" para evaluar la exactitud del modelo.

La función "train_test_split()" divide los datos en conjuntos de entrenamiento y prueba.
Los parámetros de "train_test_split()" incluyen los datos de entrada X, las etiquetas y, el tamaño del conjunto de prueba test_size y la semilla aleatoria random_state.

El método ".fit()" del pipeline entrena el modelo utilizando los datos de entrenamiento.
Los argumentos de ".fit()" son los datos de entrada X_train y las etiquetas y_train.

El método ".predict()" del pipeline genera predicciones utilizando los datos de prueba.
Los argumentos de ".predict()" son los datos de entrada X_test.

La función "accuracy_score()" calcula la exactitud del modelo comparando las predicciones con las etiquetas reales.
Los argumentos de "accuracy_score()" son las etiquetas reales y las predicciones generadas por el modelo.

La variable "pred" almacena predicciones generadas por el flujo completo.
La variable "precision" almacena la proporción de aciertos.
La evaluación mide el rendimiento del pipeline en este conjunto de prueba, que contiene solo dos ejemplos."""

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=0)
pipeline.fit(X_train, y_train)

pred = pipeline.predict(X_test)
precision = accuracy_score(y_test, pred)

print("Precisión del modelo:", precision) # Imprime la exactitud del modelo en el conjunto de prueba, aunque el texto mostrado dice "Precisión".



<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto cómo un pipeline convierte varias operaciones separadas en un flujo único, más limpio y más seguro.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">Agrupar transformaciones y modelo en una sola estructura ayuda a mantener consistencia entre entrenamiento, prueba e inferencia.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Escalar o codificar por separado fuera del flujo final y terminar aplicando pasos distintos en cada subconjunto de datos.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en el Escalado: StandardScaler y MinMaxScaler para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en el Escalado: StandardScaler y MinMaxScaler y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">02_Preprocesamiento/03_El_escalado_standard_scaler_minmax_scaler.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">